# SCOPE P2 — ML Risk Research Notebook
**Goal:** train and evaluate an Ethereum address-risk model with a held-out test set.  
**Important:** the public BLTE dataset labels *addresses* (Normal / Phishing / Scamming), not individual SCOPE actions. This notebook is an address-reputation research baseline, not a drop-in transaction classifier. Keep deterministic policy authoritative.

No model can honestly promise 99% accuracy before evaluation. This dataset is extremely imbalanced (normal addresses dominate), so accuracy alone can be misleading. Use macro-F1, minority-class recall and the confusion matrix.


In [ ]:
# Cell 1 — install dependencies if needed (run once)
%pip -q install pandas numpy scikit-learn joblib matplotlib


In [ ]:
# Cell 2 — imports, seed and paths
from pathlib import Path
import json, urllib.request, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, balanced_accuracy_score, f1_score,
                             precision_score, recall_score)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

SEED = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "raw"
ARTIFACT_DIR = ROOT / "artifacts"
REPORT_DIR = ROOT / "reports"
for p in (DATA_DIR, ARTIFACT_DIR, REPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "BLTE_multiclass_v2.csv"
MODEL_PATH = ARTIFACT_DIR / "scope_address_risk_model.joblib"
METRICS_PATH = REPORT_DIR / "scope_ml_evaluation.json"
print("Workspace:", ROOT)
print("Seed:", SEED)


In [ ]:
# Cell 3 — fetch public Ethereum BLTE multiclass CSV (no Kaggle login)
DATA_URL = "https://raw.githubusercontent.com/salam-ammari/Graph-Neural-Network-Based-Fraudulent-Address-Classification-in-Ethereum-Network/main/BLTE_multiclass_v2.csv"
if not DATA_PATH.exists() or DATA_PATH.stat().st_size < 100_000:
    req = urllib.request.Request(DATA_URL, headers={"User-Agent": "SCOPE-P2-ML-Notebook/1.0"})
    with urllib.request.urlopen(req, timeout=120) as response, open(DATA_PATH, "wb") as f:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
print(f"Dataset: {DATA_PATH} ({DATA_PATH.stat().st_size / (1024*1024):.1f} MiB)")


In [ ]:
# Cell 4 — inspect schema and class balance before modelling
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [str(c).strip() for c in df.columns]
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
label_candidates = {"class", "label", "target", "category", "risk_label"}
label_col = next((c for c in df.columns if c.strip().lower() in label_candidates), None)
if label_col is None:
    raise ValueError("Label column not detected. Inspect the displayed schema; do not guess labels.")
print("Label column:", label_col)
print(df[label_col].value_counts(dropna=False))


In [ ]:
# Cell 5 — clean and prepare numeric behavioral features
# BLTE multiclass labels: 0=Normal, 1=Phishing, 2=Scamming.
labels_numeric = pd.to_numeric(df[label_col], errors="coerce")
valid = labels_numeric.isin([0, 1, 2])
df = df.loc[valid].copy()
y = labels_numeric.loc[valid].astype(int).reset_index(drop=True)

# Exclude label and identifier-like columns to reduce leakage.
id_tokens = ("address", "wallet", "account", "hash", "txid", "node_id", "index")
drop_cols = {label_col}
for col in df.columns:
    if any(token in col.lower() for token in id_tokens):
        drop_cols.add(col)
X = df.drop(columns=list(drop_cols), errors="ignore").apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1].reset_index(drop=True)
if len(X) != len(y) or X.shape[1] < 2 or y.nunique() < 2:
    raise ValueError(f"Invalid feature matrix: X={X.shape}, y={y.shape}, classes={y.nunique()}")
print("Feature matrix:", X.shape)
print("Class counts:", y.value_counts().sort_index().to_dict())
print("Feature columns:", X.columns.tolist())


In [ ]:
# Cell 6 — stratified 60/20/20 train/validation/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print({"train": len(X_train), "validation": len(X_val), "test": len(X_test)})
print("Test class counts:", y_test.value_counts().sort_index().to_dict())


In [ ]:
# Cell 7 — train quick baselines and select by validation macro-F1
# Runtime depends on machine. RandomForest/ExtraTrees should usually be manageable on CPU.
candidates = {
    "Dummy-most-frequent": DummyClassifier(strategy="most_frequent"),
    "RandomForest-balanced": RandomForestClassifier(
        n_estimators=180, min_samples_leaf=2, class_weight="balanced_subsample",
        max_features="sqrt", random_state=SEED, n_jobs=-1
    ),
    "ExtraTrees-balanced": ExtraTreesClassifier(
        n_estimators=180, min_samples_leaf=2, class_weight="balanced",
        max_features="sqrt", random_state=SEED, n_jobs=-1
    )
}
fitted = {}
validation_rows = []
for name, estimator in candidates.items():
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", estimator)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    score = f1_score(y_val, pred, average="macro", zero_division=0)
    fitted[name] = pipe
    validation_rows.append({"model": name, "validation_macro_f1": score, "seconds": round(time.time()-t0, 2)})
validation_table = pd.DataFrame(validation_rows).sort_values("validation_macro_f1", ascending=False)
display(validation_table)
best_name = validation_table.iloc[0]["model"]
best_model = fitted[best_name]
print("Selected by validation macro-F1:", best_name)


In [ ]:
# Cell 8 — final held-out test evaluation (test set was not used for selection)
test_pred = best_model.predict(X_test)
class_names = {0:"NORMAL", 1:"PHISHING", 2:"SCAMMING"}
present_labels = sorted(int(v) for v in np.unique(y))
report = classification_report(
    y_test, test_pred, labels=present_labels,
    target_names=[class_names.get(v, f"CLASS_{v}") for v in present_labels],
    output_dict=True, zero_division=0
)
metrics = {
    "selected_model": best_name,
    "accuracy": float(accuracy_score(y_test, test_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, test_pred)),
    "macro_precision": float(precision_score(y_test, test_pred, average="macro", zero_division=0)),
    "macro_recall": float(recall_score(y_test, test_pred, average="macro", zero_division=0)),
    "macro_f1": float(f1_score(y_test, test_pred, average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(y_test, test_pred, average="weighted", zero_division=0)),
    "classification_report": report,
}
display(pd.DataFrame(metrics).drop(index="classification_report", errors="ignore").T)
print(json.dumps(metrics, indent=2))


In [ ]:
# Cell 9 — confusion matrix and per-class performance
cm = confusion_matrix(y_test, test_pred, labels=present_labels)
disp = ConfusionMatrixDisplay(cm, display_labels=[class_names.get(v, str(v)) for v in present_labels])
disp.plot(values_format="d", xticks_rotation=30)
plt.title(f"Test confusion matrix — {best_name}")
plt.tight_layout()
plt.show()
display(pd.DataFrame(report).T)


In [ ]:
# Cell 10 — refit selected model on train+validation, then save versioned artifact
# Test metrics above remain the unbiased report for the model selected on validation.
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)
best_model.fit(X_trainval, y_trainval)
bundle = {
    "model_version": "scope-address-risk-v1",
    "prediction_type": "address_reputation",
    "model": best_model,
    "feature_names": list(map(str, X.columns)),
    "label_names": {str(k): v for k, v in class_names.items()},
    "dataset_name": "BLTE multiclass Ethereum address-risk benchmark",
    "seed": SEED,
    "warning": "Address-level prediction, not direct SCOPE action-level fraud prediction."
}
joblib.dump(bundle, MODEL_PATH)
result = {
    "model_version": bundle["model_version"],
    "dataset": bundle["dataset_name"],
    "rows_used": int(len(X)),
    "feature_count": int(X.shape[1]),
    "class_counts": {str(int(k)): int(v) for k,v in y.value_counts().sort_index().items()},
    "split_sizes": {"train":len(X_train), "validation":len(X_val), "test":len(X_test)},
    "metrics": metrics,
    "confusion_matrix_labels": present_labels,
    "confusion_matrix": cm.tolist(),
    "limitations": [
        "Address labels are not transaction/action labels.",
        "Random stratified split is not temporal/external validation.",
        "High accuracy can be misleading because classes are severely imbalanced.",
        "ML is advisory; deterministic hard-deny policy must remain authoritative."
    ]
}
METRICS_PATH.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH)
print("Saved metrics:", METRICS_PATH)


In [ ]:
# Cell 11 — smoke-test the saved artifact and show integration contract
loaded = joblib.load(MODEL_PATH)
assert loaded["model_version"] == "scope-address-risk-v1"
assert len(loaded["feature_names"]) == X.shape[1]
print("Artifact load test: PASS")
print("Model version:", loaded["model_version"])
print("Feature count:", len(loaded["feature_names"]))
print("Required inference feature names:", loaded["feature_names"])
print("\nDo NOT pass action fields as if they were these address-level features.")


## Integration boundary (read before connecting to FastAPI)
- This notebook produces an **address reputation** model.
- It cannot be wired directly to the current action schema unless a reliable feature provider supplies the exact model input features for the target address.
- Do not invent missing features, infer labels from your existing deterministic rules, or claim the test score generalizes to unseen future attacks.
- Deterministic policy must retain authority: explicit DENY rules cannot be overridden by ML.
- Never report 99% as achieved unless the held-out metrics actually show it, and even then report minority-class recall and the confusion matrix.
